In [1]:
from google.cloud import bigquery
from pathlib import Path
import os
import requests
import pandas as pd
import db_dtypes
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import model_config 
import importlib

In [2]:
def load_data():

    path = Path("D:\Python Projects\Hospital readmission risk\index_stay.csv")

    if path.exists():
        return pd.read_csv(path, sep = ',')


    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"D:\Python Projects\Hospital readmission risk\.secrets\hospital-readmission-4-code.json"
    client = bigquery.Client(project = "hospital-readmission-4")
    sql = """
    SELECT
        *
        from `hospital-readmission-4.helper_tables.index_stay`
    """

    job = client.query(sql)
    rows = list(job.result())


    data_raw = [dict(r) for r in rows]
    return pd.DataFrame(data_raw)

In [3]:
def build_preprocessor(df_raw):

    analysis_cols = [
    'patient_age', 'gender', 'length_of_stay', 'main_code', 'num_diagnoses',
    'num_chronic_conditions', 'num_procedures', 'has_diabetes', 'has_cancer',
    'has_hiv', 'has_hf', 'has_alz', 'has_ckd', 'had_surgery', 'admission_cost',
    'total_procedure_costs', 'total_medication_costs', 'total_stay_cost', 
    'admissions_365d', 'tot_length_of_stay_365d', 'avg_cost_of_prev_stays',
    'is_planned', 'following_unplanned_admission_flag', 'readmit_30d', 'readmit_90d'
    ]

    df_numeric = df_raw[analysis_cols]

    df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
    df_numeric['following_unplanned_admission_flag'] = df_numeric['following_unplanned_admission_flag'].fillna(0)

    df_numeric = pd.get_dummies(df_numeric)
    df_numeric = df_numeric.drop(columns = 'gender_F')

    mask = df_numeric['following_unplanned_admission_flag'] == 0
    df_numeric.loc[mask, ['readmit_30d', 'readmit_90d']] = 0
    mask = df_numeric['readmit_90d'] == 0
    df_numeric.loc[mask, 'following_unplanned_admission_flag'] = 0

    df_results = df_numeric[['readmit_30d', 'readmit_90d']]
    df_numeric['log_stay_cost'] = np.log(df_numeric['total_stay_cost'])
    df_numeric['log_prev_avg_stay_cost'] = np.log1p(df_numeric['avg_cost_of_prev_stays'])
    df_numeric['log_total_procedure_costs'] = np.log1p(df_numeric['total_procedure_costs'])
    df_numeric['log_total_medication_costs'] = np.log1p(df_numeric['total_medication_costs'])

    df_numeric = df_numeric.drop(columns = ['total_stay_cost', 'avg_cost_of_prev_stays', 'total_procedure_costs', 'total_medication_costs'
,'readmit_30d', 'readmit_90d', 'following_unplanned_admission_flag', 'main_code'])

    return df_numeric, df_results

In [4]:
def make_train_test_split(X, y):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

    return X_train, X_test, y_train, y_test

In [5]:
def train_model(X, y, model_name, model, params, test_log, d30 = True):

    model_name = model_name + ('_d30' if d30 else '_d90')

    model = model.set_params(**params)

    pipe = Pipeline([('scaler', StandardScaler()), (model_name, model)])
    """
    scoring = ["roc_auc", "average_precision"]

    cv_results = cross_validate(
        estimator=pipe,
        X=X,
        y=y,
        cv=5,
        scoring=scoring,
        return_train_score=False,
    )

    test_log.loc[model_name, 'roc_auc'] = cv_results["test_roc_auc"].mean()
    test_log.loc[model_name, 'roc_auc_std'] = cv_results["test_roc_auc"].std()
    test_log.loc[model_name, 'pr_auc'] = cv_results["test_average_precision"].mean()
    test_log.loc[model_name, 'pr_auc_std'] = cv_results["test_average_precision"].std()
    """
    pipe.fit(X,y)
    
    return pipe, model_name, test_log
    

In [6]:
def evaluate_model(X, y_true, model_name, pipe, coefs, metrics, pred):


    y_proba = pipe.predict_proba(X)[:,1]
    y_pred =  pipe.predict(X)

    metrics.loc[model_name, 'roc'] = roc_auc_score(y_true, y_proba)
    metrics.loc[model_name, 'pr'] = average_precision_score(y_true, y_proba)
    metrics.loc[model_name, 'brier_loss_total'] = brier_score_loss(y_true, y_proba)
    metrics.loc[model_name, 'brier_loss_half'] = brier_score_loss(y_true, np.array(y_proba) > 0.5)

    precision, recall, f1, _= precision_recall_fscore_support(y_true, y_pred, average="binary")
    metrics.loc[model_name, 'precision'] = precision
    metrics.loc[model_name, 'recall'] = recall
    metrics.loc[model_name, 'f1'] = f1

    pred[model_name] = y_proba

    est = pipe.named_steps[model_name]

    if isinstance(est, LogisticRegression):
    # est.coef_.shape == (1, n_features) for binary
        coefs[model_name] = est.coef_[0]

    elif hasattr(est, "feature_importances_"):
    # trees, random forest, gradient boosting
        coefs[model_name] = est.feature_importances_

    return coefs, metrics, pred



In [115]:
def build_threshold_metrics(values):

    true_values = values[['readmit_30d', 'readmit_90d']]

    values = values.drop(columns = ['readmit_30d', 'readmit_90d'])

    thresholds = pd.DataFrame(index = values.index)

    metrics_index = ['TP', 'FP', 'FN', 'TN', 'precision', 'recall']

    metrics = pd.DataFrame(index = metrics_index)

    for model in values.columns:

        for t in [round(t, 2) for t in np.arange(0.05, 1, 0.05)]:

            thresholds[model + '_' + str(t)] = (values[model] >= t).astype(int)

    thresholds['readmit_30d'] = true_values['readmit_30d']
    thresholds['readmit_90d'] = true_values['readmit_90d']

    for model_threshold in thresholds.columns:

        if(model_threshold not in ['readmit_30d', 'readmit_90d']):

            true_col = ('readmit_30d' if 'd30' in model_threshold else 'readmit_90d')

            metrics.loc['TP', model_threshold] = ((thresholds[model_threshold] == 1) & (thresholds[true_col] == 1)).sum()
            metrics.loc['FP', model_threshold] = ((thresholds[model_threshold] == 1) & (thresholds[true_col] == 0)).sum()
            metrics.loc['FN', model_threshold] = ((thresholds[model_threshold] == 0) & (thresholds[true_col] == 1)).sum()
            metrics.loc['TN', model_threshold] = ((thresholds[model_threshold] == 0) & (thresholds[true_col] == 0)).sum()

            metrics.loc['precision', model_threshold] = metrics.loc['TP', model_threshold]/(metrics.loc['TP', model_threshold] + metrics.loc['FP', model_threshold])
            metrics.loc['recall', model_threshold] = metrics.loc['TP', model_threshold]/(metrics.loc['TP', model_threshold] + metrics.loc['FN', model_threshold])


    return thresholds, metrics

In [98]:
def cost_reduction_preprocessor(df_raw, df_pred, df_flag):

    cols_30 = [col for col in df_pred if 'd30' in col]

    cols = ['stay_id','length_of_stay','cost_per_day_stay','total_readmission_cost', 'avg_cost_of_prev_stays'] 
    rows = df_pred.index

    df = df_raw.loc[rows, cols]
    df['readmit_30d'] = df_flag.loc[rows, 'readmit_30d']
    
    df['total_readmission_cost'] = df['total_readmission_cost'].fillna(0)

    df[cols_30] = df_pred[cols_30]

    return df



In [93]:
def estimate_cost_reduction(df_cost, pred_values, r = 0.2):

    gains = pd.DataFrame(index = df_cost.index)

    for col in df_cost.columns:

        if ('d30' in col):

            pos = col.rfind("_")
            model = col[:pos]
            threshold = float(col[(pos + 1):])

            for id in df_cost.index:

                if(df_cost.loc[id, col] == 1):

                    exp_avoided_cost = r * pred_values.loc[id, model] * df_cost.loc[id, 'total_readmission_cost']

                    relative_prob = pred_values.loc[id, model] / threshold

                    if(relative_prob < 2 * threshold):

                        intervention_cost = min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])

                    elif(relative_prob < 3 * threshold):

                        intervention_cost = 2 * min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])
                    
                    else:

                        intervention_cost = 3 * min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])

                    gains.loc[id, col] = exp_avoided_cost - intervention_cost

                else: 

                    gains.loc[id, col] = 0

    gains.loc['total_avoided'] = gains.sum(axis = 0)

    return gains

In [8]:
df_raw = load_data()
df_numeric, df_results = build_preprocessor(df_raw)

C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\2591555852.py:6: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ',')
C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\2235083803.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\2235083803.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a

In [ ]:
models = {
    'logreg': LogisticRegression(),
    'rf': RandomForestClassifier(),
    'lightgbm': LGBMClassifier()
}

test_log = pd.DataFrame(columns = ['roc_auc', 'pr_auc'])

metrics_log = pd.DataFrame(columns = ['roc', 'pr', 'brier_loss_total', 'brier_loss_half', 'precision', 'recall', 'f1'])

params = {'class_weight': 'balanced', 'solver': 'saga', 'max_iter': 2000}

coefs = pd.DataFrame(index = df_numeric.columns)

pred_values = pd.DataFrame()

for model_name in models:

    for set in model_config.MODEL_CONFIGS:

        if set['name'] == model_name:

            params = set['params']

    X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
    
    trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log)

    if pred_values.shape[0] < 2:

        pred_values['X'] = X_test.index

        pred_values.set_index('X', inplace = True)

    if pred_values.shape[1] < 1:

        pred_values['readmit_30d'] = y_test

    coefs, metrics_log, pred_values = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log, pred_values)

    X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])
    
    trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log, d30 = False)

    if pred_values.shape[1] < 3:

        pred_values['readmit_90d'] = y_test

    coefs, metrics_log, pred_values = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log, pred_values)



[LightGBM] [Info] Number of positive: 4119, number of negative: 83483
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047019 -> initscore=-3.009033
[LightGBM] [Info] Start training from score -3.009033


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8044, number of negative: 79558
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.091824 -> initscore=-2.291560
[LightGBM] [Info] Start training from score -2.291560


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [116]:
thresholds, threshold_metrics = build_threshold_metrics(pred_values)

df_cost = cost_reduction_preprocessor(df_raw, thresholds, df_results)

gains = estimate_cost_reduction(df_cost, pred_values)
    

C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\1642686626.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  thresholds[model + '_' + str(t)] = (values[model] >= t).astype(int)
C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\1642686626.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  thresholds[model + '_' + str(t)] = (values[model] >= t).astype(int)
C:\Users\4infi\AppData\Local\Temp\ipykernel_13140\1642686626.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

In [117]:
gains

,logreg_d30_0.05,logreg_d30_0.1,logreg_d30_0.15,logreg_d30_0.2,logreg_d30_0.25,logreg_d30_0.3,logreg_d30_0.35,logreg_d30_0.4,logreg_d30_0.45,logreg_d30_0.5,...,lightgbm_d30_0.5,lightgbm_d30_0.55,lightgbm_d30_0.6,lightgbm_d30_0.65,lightgbm_d30_0.7,lightgbm_d30_0.75,lightgbm_d30_0.8,lightgbm_d30_0.85,lightgbm_d30_0.9,lightgbm_d30_0.95
X,,,,,,,,,,,,,,,,,,,,,
108992,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,-5.147220e+04,...,-3.422831e+04,-3.422831e+04,-1.175611e+04,-1.175611e+04,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
52054,-7.706910e+03,-7.706910e+03,-7.706910e+03,-7.706910e+03,-7.706910e+03,-7.706910e+03,-7.706910e+03,-7.706910e+03,-5.137940e+03,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
466,-1.252200e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
6985,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
27343,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,-6.222210e+03,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17190,-5.905410e+03,-5.905410e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
16716,-1.690266e+04,-1.690266e+04,-1.690266e+04,-1.690266e+04,-1.690266e+04,-1.690266e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
41507,-1.909788e+04,-1.909788e+04,-1.909788e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00


In [118]:
positive_model_mask = gains.loc['total_avoided'] > 0

gains.loc[:, positive_model_mask]

,logreg_d30_0.7,logreg_d30_0.75,logreg_d30_0.8,logreg_d30_0.85,logreg_d30_0.9,logreg_d30_0.95,rf_d30_0.75,rf_d30_0.8,rf_d30_0.85,rf_d30_0.9
X,,,,,,,,,,
108992,-29000.003828,-6.527804e+03,-6.527804e+03,-6.527804e+03,-6.527804e+03,-6.527804e+03,-10026.540528,0.000000,0.000000,0.000000
52054,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
466,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
6985,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
27343,-2074.070000,-2.074070e+03,-2.074070e+03,-2.074070e+03,-2.074070e+03,-2.074070e+03,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
17190,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
16716,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
41507,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000


In [119]:
cols_30 = [col for col in threshold_metrics if 'd30' in col]
threshold_30 = threshold_metrics[cols_30]

In [120]:
threshold_30.loc[:, positive_model_mask]

,logreg_d30_0.7,logreg_d30_0.75,logreg_d30_0.8,logreg_d30_0.85,logreg_d30_0.9,logreg_d30_0.95,rf_d30_0.75,rf_d30_0.8,rf_d30_0.85,rf_d30_0.9
TP,770.000000,749.000000,728.000000,695.000000,616.000000,412.000000,477.000000,418.000000,368.000000,307.000000
FP,1158.000000,1047.000000,948.000000,829.000000,661.000000,380.000000,226.000000,139.000000,86.000000,53.000000
FN,272.000000,293.000000,314.000000,347.000000,426.000000,630.000000,565.000000,624.000000,674.000000,735.000000
TN,19701.000000,19812.000000,19911.000000,20030.000000,20198.000000,20479.000000,20633.000000,20720.000000,20773.000000,20806.000000
precision,0.399378,0.417038,0.434368,0.456037,0.482381,0.520202,0.678521,0.750449,0.810573,0.852778
recall,0.738964,0.718810,0.698656,0.666987,0.591171,0.395393,0.457774,0.401152,0.353167,0.294626


In [122]:
rows = pred_values[pred_values['readmit_30d'] == 1].index

cost_saved = gains.loc['total_avoided', 'logreg_d30_0.85'] / df_raw.loc[rows, 'total_readmission_cost'].sum()

In [123]:
cost_saved

np.float64(0.050395237343900605)